In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import KFold, cross_val_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings

warnings.filterwarnings('ignore')

def get_processed_data():
    # Original Claims Data
    claims = pd.DataFrame({
        'Month': [1, 2, 3, 4],
        'AC': [874, 2314, 6320, 12001],
        'TV': [2198, 1912, 2443, 2206],
        'HA_Speaker': [16545 + 211, 12908 + 178, 13902 + 176, 13730 + 181], # Merged HA & Speaker
        'Total_Claims': [19828, 17312, 22841, 28118]
    })

    # Original Contacts Data
    contacts_raw = pd.DataFrame({
        'Month': [1, 2, 3, 4],
        'AC': [2769, 5475, 15324, 33778],
        'TV': [6430, 4792, 6053, 5217],
        'HA_Speaker': [43872 + 581, 35155 + 510, 38685 + 590, 38569 + 566], # Merged HA & Speaker
        'Enquiry': [11250, 9722, 14771, 23892],
        'Total_Contacts': [64902, 55654, 75423, 102022]
    })

    
    contacts_raw['Enquiry_Split'] = contacts_raw['Enquiry'] / 3
    contacts_raw['AC'] += contacts_raw['Enquiry_Split']
    contacts_raw['TV'] += contacts_raw['Enquiry_Split']
    contacts_raw['HA_Speaker'] += contacts_raw['Enquiry_Split']

    return claims, contacts_raw

df_claims, df_contacts = get_processed_data()


df = pd.merge(df_claims[['Month', 'Total_Claims']], 
              df_contacts[['Month', 'Total_Contacts', 'AC', 'HA_Speaker']], on='Month')

df['overall_ctu'] = (df['Total_Contacts'] / df['Total_Claims']) * 100
df['ac_mix_ratio'] = df['AC'] / df['Total_Contacts']
df['ha_audio_mix'] = df['HA_Speaker'] / df['Total_Contacts']

X = df[['Month', 'ac_mix_ratio', 'ha_audio_mix']]
y = df['overall_ctu']


def run_model_comparison(X, y):
    results = []
    
    X_stat = sm.add_constant(X)
    lr_stats = sm.OLS(y, X_stat).fit()
    p_val_ac = lr_stats.pvalues['ac_mix_ratio']
    lr_preds = lr_stats.predict(X_stat)

    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X, y)
    rf_preds = rf.predict(X)

    xgb = GradientBoostingRegressor(n_estimators=100, random_state=42)
    xgb.fit(X, y)
    xgb_preds = xgb.predict(X)

    sarima = SARIMAX(y, exog=X[['ac_mix_ratio']], order=(1,0,0)).fit(disp=False)
    sarima_preds = sarima.predict(start=0, end=len(y)-1, exog=X[['ac_mix_ratio']])

    # 5. K-Fold Cross Validation
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    cv_scores = cross_val_score(xgb, X, y, cv=kf, scoring='neg_mean_absolute_percentage_error')
    cv_stability = (1 + np.mean(cv_scores)) * 100

    model_data = [('Linear Regression', lr_preds), ('Random Forest', rf_preds), 
                  ('XGBoost Proxy', xgb_preds), ('SARIMA', sarima_preds)]
    
    for name, pred in model_data:
        results.append({
            'Model': name,
            'MAPE (%)': mean_absolute_percentage_error(y, pred) * 100,
            'RMSE': np.sqrt(mean_squared_error(y, pred)),
            'P-Value (AC)': f"{p_val_ac:.4f}" if name == 'Linear Regression' else "N/A"
        })
    
    return pd.DataFrame(results), cv_stability

# --- STEP 3: OUTPUT ---
summary_df, stability_score = run_model_comparison(X, y)

print("--- 2026 CTU PROOF (Redistributed Enquiry & Merged HA/Speaker) ---")
print(summary_df.to_string(index=False))
print(f"\nModel Stability (K-Fold CV): {stability_score:.2f}%")
print(f"Significant AC Trend (P < 0.05): {'YES' if float(summary_df.iloc[0]['P-Value (AC)']) < 0.05 else 'No'}")

--- 2026 CTU PROOF (Redistributed Enquiry & Merged HA/Speaker) ---
            Model     MAPE (%)         RMSE P-Value (AC)
Linear Regression 1.078981e-13 4.167441e-13          nan
    Random Forest 1.020837e+00 5.244426e+00          N/A
    XGBoost Proxy 1.060532e-04 4.280208e-04          N/A
           SARIMA 2.633928e+01 1.564040e+02          N/A

Model Stability (K-Fold CV): 96.19%
Significant AC Trend (P < 0.05): No
